# Queue-Reactive Model Validation

This notebook calibrates and compares four Queue-Reactive model variants on the reconstructed event-flow dataset:

- `QR`: baseline model with empirical unconditional size distribution
- `QRU`: unit-size version using `ceil(AES)`
- `FTQR`: five-type model with full-queue cancel and market events
- `SAQR`: size-aware model using `lambda_(eta, s)(n)`

The calibration uses the timestamp-corrected state-duration estimator at level 1 only.

In [ ]:
from pathlib import Path
import sys
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import models.common as common_module
import models.qr as qr_module
import models.qru as qru_module
import models.ftqr as ftqr_module
import models.saqr as saqr_module

importlib.reload(common_module)
importlib.reload(qr_module)
importlib.reload(qru_module)
importlib.reload(ftqr_module)
importlib.reload(saqr_module)

from models.common import calibrate_common
from models.qr import QRSimulator, calibrate_qr
from models.qru import QRUSimulator, calibrate_qru
from models.ftqr import FTQRSimulator, calibrate_ftqr
from models.saqr import SAQRSimulator, calibrate_saqr


In [ ]:
EVENT_FLOW_PATH = ROOT / "data/processed/FGBL_event_flow.parquet"
RAW_DIR = ROOT / "data/raw"
LEVEL = 1
MIN_OBS = 50
SIM_STEPS = 25_000

common = calibrate_common(
    event_flow_path=str(EVENT_FLOW_PATH),
    raw_dir=str(RAW_DIR),
    level=LEVEL,
    min_obs=MIN_OBS,
)

qr_cal = calibrate_qr(str(EVENT_FLOW_PATH), raw_dir=str(RAW_DIR), level=LEVEL, min_obs=MIN_OBS, common=common)
qru_cal = calibrate_qru(str(EVENT_FLOW_PATH), raw_dir=str(RAW_DIR), level=LEVEL, min_obs=MIN_OBS, common=common)
ftqr_cal = calibrate_ftqr(str(EVENT_FLOW_PATH), raw_dir=str(RAW_DIR), level=LEVEL, min_obs=MIN_OBS, common=common)
saqr_cal = calibrate_saqr(str(EVENT_FLOW_PATH), raw_dir=str(RAW_DIR), level=LEVEL, min_obs=MIN_OBS, smoothing_alpha=25.0, common=common)

summary_df = pd.DataFrame(
    {
        "level": [common.level],
        "aes_level": [common.aes_level],
        "n_min": [int(common.intensity_df["n"].min())],
        "n_max": [int(common.intensity_df["n"].max())],
        "n_bins": [len(common.intensity_df)],
        "size_support": [len(common.size_distribution)],
        "joint_support": [len(common.joint_size_df)],
        "qru_unit_size": [qru_cal.unit_size],
    }
)

display(summary_df)
display(common.intensity_df.head(10))


## Figure 3 Style: Baseline QR Intensities

These are the level-1 intensity curves from the timestamp-corrected state-duration calibration.

In [ ]:
curve_df = common.intensity_df.copy().sort_values("n")
plot_cols = ["lambda_L", "lambda_C", "lambda_M"]
titles = ["lambda_L(n)", "lambda_C(n)", "lambda_M(n)"]
colors = ["#1f77b4", "#d62728", "#2ca02c"]
window = 5
n_plot_max = min(100, int(curve_df["n"].max()))

fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=True)
for ax, col, title, color in zip(axes, plot_cols, titles, colors):
    plot_df = curve_df[curve_df["n"] <= n_plot_max].copy()
    plot_df[f"{col}_smooth"] = plot_df[col].rolling(window=window, min_periods=1, center=True).mean()
    ax.plot(plot_df["n"], plot_df[col], color=color, alpha=0.25)
    ax.plot(plot_df["n"], plot_df[f"{col}_smooth"], color=color, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Queue size (AES units)")
    ax.grid(alpha=0.2)
axes[0].set_ylabel("Intensity")
fig.suptitle("QR Intensities at Level 1", y=1.02)
fig.tight_layout()


## FTQR: Partial vs Full-Queue Removal Intensities

FTQR separates partial and queue-clearing cancel/trade events.

In [ ]:
ft_df = ftqr_cal.intensity_df.copy().sort_values("n")
n_plot_max_ft = min(100, int(ft_df["n"].max()))
plot_df = ft_df[ft_df["n"] <= n_plot_max_ft].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
for ax, partial_col, full_col, title in [
    (axes[0], "lambda_C", "lambda_C_all", "Cancels"),
    (axes[1], "lambda_M", "lambda_M_all", "Trades"),
]:
    ax.plot(plot_df["n"], plot_df[partial_col], linewidth=2, label=partial_col)
    ax.plot(plot_df["n"], plot_df[full_col], linewidth=2, label=full_col)
    ax.set_title(title)
    ax.set_xlabel("Queue size (AES units)")
    ax.grid(alpha=0.2)
    ax.legend()
axes[0].set_ylabel("Intensity")
fig.suptitle("FTQR Partial vs Full-Queue Intensities", y=1.02)
fig.tight_layout()


## SAQR Heatmaps

The heatmaps below show `lambda_(eta, s)(n)` for the three base event types.

In [ ]:
joint_df = saqr_cal.joint_size_df.copy()
joint_df = joint_df[(joint_df["n"] <= 60) & (joint_df["size"] <= 150)].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
for ax, eta in zip(axes, ["L", "C", "M"]):
    heat = joint_df[joint_df["eta"] == eta].pivot_table(
        index="n",
        columns="size",
        values="lambda_eta_size",
        aggfunc="mean",
        fill_value=0.0,
    )
    im = ax.imshow(
        np.log1p(heat.to_numpy()),
        aspect="auto",
        origin="lower",
        cmap="viridis",
        extent=[heat.columns.min(), heat.columns.max(), heat.index.min(), heat.index.max()],
    )
    ax.set_title(f"SAQR {eta}")
    ax.set_xlabel("Event size")
    ax.grid(False)
axes[0].set_ylabel("Queue size n (AES)")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.9, label="log(1 + intensity)")
fig.suptitle("SAQR Heatmaps", y=1.02)
fig.tight_layout()


## Simulation Comparison

Short synthetic runs under the four calibrated models.

In [ ]:
simulators = {
    "QR": QRSimulator(qr_cal),
    "QRU": QRUSimulator(qru_cal),
    "FTQR": FTQRSimulator(ftqr_cal),
    "SAQR": SAQRSimulator(saqr_cal),
}

sim_results = {
    name: simulator.simulate(steps=SIM_STEPS, seed=i + 1)
    for i, (name, simulator) in enumerate(simulators.items())
}

sim_summary = pd.DataFrame({name: result.summary for name, result in sim_results.items()}).T
display(sim_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sim_summary["depletion_rate"].plot(kind="bar", ax=axes[0], color="#1f77b4")
axes[0].set_title("Simulated Depletion Rate")
axes[0].set_ylabel("Rate")
axes[0].grid(axis="y", alpha=0.2)

sim_summary["price_vol"].plot(kind="bar", ax=axes[1], color="#d62728")
axes[1].set_title("Simulated Price Volatility")
axes[1].set_ylabel("Std of price increments")
axes[1].grid(axis="y", alpha=0.2)

fig.tight_layout()
